In [2]:
!pip install torchtext

from torchtext.data import Field, LabelField
from torchtext.datasets import IMDB

TEXT = Field(sequential=True, lower=True, batch_first=True)
LABEL = LabelField(dtype=None)

# Load the dataset by passing the fields
train_dataset, test_dataset = IMDB.splits(TEXT, LABEL)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 3.5 MB/s eta 0:00:00
downloading aclImdb_v1.tar.gz


aclImdb_v1.tar.gz: 100%|██████████| 84.1M/84.1M [00:01<00:00, 49.8MB/s]


### preprocessing steps before feeding the dataset to rnn:
1. Split the training dataset into separate training and validation partitions.
2. Identify the unique words in the training dataset
3. Map each unique word to a unique integer and encode the review text into encoded integers (an index of each unique word)
4. Divide the dataset into mini-batches as input to the model

In [3]:
#1st step: creating the dataset for training and validation
from torch.utils.data.dataset import random_split
import torch
torch.manual_seed(1)

# Convert to a list to get its stable size
train_list = list(train_dataset)
total_len = len(train_list)
train_len = int(0.8 * total_len)
valid_len = total_len - train_len

# Perform split using dynamic lengths
train_split, valid_split = random_split(train_list, [train_len, valid_len])

# finding uniques codes 2nd step

import re
import collections
from collections import Counter, OrderedDict

def tokenizer(text):
  text = re.sub('<[^>]>', '', text)
  emoticoins = re.findall(r'(?::|;| = )(?:-)?(?:\)|"|\(|D|P)', text.lower())
  text = re.sub(r'[\W] + ', ' ', text.lower()) + ' '.join(emoticoins).replace('-', '')
  tokenized = text.split()
  return tokenized

tokens_counts = Counter()
for example in train_split:
  tokens = tokenizer(" ".join(example.text))
  tokens_counts.update(tokens)
print('vocab-size: ', len(tokens_counts))
print(f'total len {total_len}')

vocab-size:  220141
total len 25000


In [4]:
# 3rd step: encoding each token to tokens through vocab
# how Vocab works in torchtext 0.6.0:
from torchtext.vocab import Vocab

# Pass the Counter object directly, and define special tokens in the constructor
vocab = Vocab(tokens_counts, specials=['<pad>', '<unk>'])

# In torchtext 0.6.0, <unk> is at index 0 by default.
# Let's verify how vocab works on sample tokens:
# Use .stoi (string-to-index) to look up indices, defaulting to the <unk> index (0) if not found
print("Index of '<pad>':", vocab.stoi['<pad>'])
print("Index of '<unk>':", vocab.stoi['<unk>'])
print("Index of 'this':", vocab.stoi['this'])

# How to safely look up tokens with a fallback for unknown words:
unk_idx = vocab.stoi['<unk>']
sample_indices = [vocab.stoi.get(token, unk_idx) for token in ['this', 'is', 'a', 'example']]
print("Encoded sample:", sample_indices)

Index of '<pad>': 0
Index of '<unk>': 1
Index of 'this': 10
Encoded sample: [10, 7, 3, 682]


In [5]:
# defining function of tokenizer
import torch
import torch.nn as nn

text_pipeline = lambda x: [vocab[token] for token in tokenizer(x)]
label_pipeline = lambda x: 1 if x == 'pos' else 0

# wrap the encode and transformation batch :
def collate_batch(batch):
  lengths, label_list, text_list = [], [], []

  for _label, _text in batch:
    label_list.append(label_pipeline(_label))
    processed_text = torch.tensor(text_pipeline(_text), dtype=torch.int64)
    text_list.append(processed_text)
    lengths.append(processed_text.size(0))

  # These must be computed OUTSIDE the for loop
  padded_text_list = nn.utils.rnn.pad_sequence(text_list, batch_first=True, padding_value=vocab.stoi['<pad>'])
  lengths = torch.tensor(lengths)
  label_list = torch.tensor(label_list)

  return padded_text_list, lengths, label_list

In [18]:
# We pass the standard Python list of tuples to the DataLoader
from torch.utils.data import DataLoader

# Updated collate_batch to correctly access Example attributes in torchtext 0.6.0
def collate_batch(batch):
  lengths, label_list, text_list = [], [], []

  for example in batch:
    # Access the label and text attributes from the Example object
    label_list.append(label_pipeline(example.label))
    processed_text = torch.tensor(text_pipeline(" ".join(example.text)), dtype=torch.int64)
    text_list.append(processed_text)
    lengths.append(processed_text.size(0))

  padded_text_list = nn.utils.rnn.pad_sequence(text_list, batch_first=True, padding_value=vocab.stoi['<pad>'])
  lengths = torch.tensor(lengths)
  label_list = torch.tensor(label_list)

  return padded_text_list, lengths, label_list

dataloader = DataLoader(train_split, batch_size=4, shuffle=False, collate_fn=collate_batch)

text_batch, lengths_batch, label_batch = next(iter(dataloader))
print("Text batch shape:", text_batch.shape)
print("Lengths:", lengths_batch)
print("Labels:", label_batch)

# Convert test_dataset to list so that it supports random indexing in the DataLoader
test_list = list(test_dataset)

batch_size = 32
train_dl = DataLoader(train_split, batch_size=batch_size, shuffle=True, collate_fn=collate_batch)
valid_dl = DataLoader(valid_split, batch_size=batch_size,  shuffle=False, collate_fn=collate_batch)
test_dl = DataLoader(test_list, batch_size=batch_size, shuffle=False, collate_fn=collate_batch)

Text batch shape: torch.Size([4, 291])
Lengths: tensor([112, 291, 175, 144])
Labels: tensor([0, 0, 0, 1])


# Embedding layers for sentence encoding
in embedding it takes a vocabulary i.e 2 and then maps random vectors in any dimension then using backpropagation to calculate the overshooting like a graph of x,y,z, then using optimizer to update those overshooting and undershooting

In [7]:
embedding = nn.Embedding(num_embeddings = 10, embedding_dim = 3, padding_idx = 3)
# the embedding is [3,10] dimension i.e a word lie "aron" would be [[ 0.2161, -1.2073,  0.8810]] where 10 is the size of vocabulary
# for example two sentences
text_encoded_inpurt = torch.LongTensor([[1,2,3,4], [5,4,5,6]])
print(embedding(text_encoded_inpurt))

tensor([[[ 0.7039, -0.8321, -0.4651],
         [-0.3203,  2.2408,  0.5566],
         [ 0.0000,  0.0000,  0.0000],
         [-0.4643,  0.3046,  0.7046]],

        [[-0.7106, -0.2959,  0.8356],
         [-0.4643,  0.3046,  0.7046],
         [-0.7106, -0.2959,  0.8356],
         [ 0.4025, -0.6924, -0.4442]]], grad_fn=<EmbeddingBackward0>)


# Building an RNN model

In [8]:
class RNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()
        # Embedding layer to convert integer indices to dense vectors
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=vocab.stoi['<pad>'])
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True, num_layers=2)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x, lengths=None):
        # x shape: (batch_size, seq_len)
        embedded = self.embedding(x)
        # embedded shape: (batch_size, seq_len, embedding_dim)
        output, hn = self.rnn(embedded)
        # Select the last valid output step for each batch item
        out = output[:, -1, :]
        out = torch.sigmoid(self.fc(out))
        return out

# Instantiate the model with correct vocab size
model = RNN(len(vocab), 64, 32)
print(model)

RNN(
  (embedding): Embedding(220143, 64, padding_idx=0)
  (rnn): RNN(64, 32, num_layers=2, batch_first=True)
  (fc): Linear(in_features=32, out_features=1, bias=True)
)


In [9]:
def train(dataloader):
  model.train()
  total_acc, total_loss = 0, 0
  for text_batch, lengths_batch, label_batch in dataloader :
    optimizer.zero_grad()
    pred = model(text_batch, lengths_batch)[:, 0]
    # Cast label_batch to float to match the expected dtype of BCELoss
    loss = loss_fn(pred, label_batch.float())
    loss.backward()
    optimizer.step()
    total_acc += ((pred >= 0.5) == label_batch).sum().item()
    total_loss += loss.item() * label_batch.size(0)
  return total_acc / len(dataloader.dataset), total_loss / len(dataloader.dataset)

def evaluate(dataloader):
  model.eval()
  total_acc, total_loss = 0, 0
  with torch.no_grad():
    for text_batch, lengths_batch, label_batch in dataloader :
      pred = model(text_batch, lengths_batch)[:, 0]
      # Cast label_batch to float to match the expected dtype of BCELoss
      loss = loss_fn(pred, label_batch.float())
      total_acc += ((pred >= 0.5) == label_batch).sum().item()
      total_loss += loss.item() * label_batch.size(0)

  return total_acc / len(dataloader.dataset), total_loss / len(dataloader.dataset)

In [10]:
loss_fn = nn.BCELoss()

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [15]:
num_epochs = 10
torch.manual_seed(1)
for epoch in range(num_epochs):
    acc_train, loss_train = train(train_dl)
    acc_valid, loss_valid = evaluate(valid_dl)
    print(f'Epoch {epoch} accuracy: {acc_train:.4f} val_accuracy: {acc_valid:.4f}')

Epoch 0 accuracy: 0.5104 val_accuracy: 0.4906
Epoch 1 accuracy: 0.5093 val_accuracy: 0.4962
Epoch 2 accuracy: 0.5058 val_accuracy: 0.5100
Epoch 3 accuracy: 0.5162 val_accuracy: 0.4964
Epoch 4 accuracy: 0.5113 val_accuracy: 0.5020
Epoch 5 accuracy: 0.5093 val_accuracy: 0.4986
Epoch 6 accuracy: 0.5181 val_accuracy: 0.4966
Epoch 7 accuracy: 0.5126 val_accuracy: 0.4876
Epoch 8 accuracy: 0.5182 val_accuracy: 0.4958
Epoch 9 accuracy: 0.5148 val_accuracy: 0.4960


In [19]:
acc_test, _ = evaluate(test_dl)
print(f'test_accuracy: {acc_test:.4f}')

test_accuracy: 0.4989
